# 📰 News Bias & Fact Clustering Engine

An AI system that pulls the **same real-world news event as covered by multiple outlets**,
groups those articles together using semantic clustering, and then compares
**how differently each outlet framed the story** — tone, sentiment, subjectivity, and keyword choice.

This is *not* a fake-news detector and it does **not** label anything as "true" or "false."
It answers a narrower, more useful question:

> *"Different outlets covered the same event — how did their language and framing differ?"*

## Features
- Pulls live articles from multiple news outlets via RSS (no paid API key needed)
- Groups articles that are about the *same underlying event* using sentence embeddings + clustering
- Scores each article's sentiment & subjectivity (TextBlob)
- Extracts framing keywords per outlet with KeyBERT
- Side-by-side "event report" comparing outlets on the same story
- Interactive Gradio UI to explore clusters

## Tech Stack
- `feedparser` — RSS ingestion
- `sentence-transformers` (`all-MiniLM-L6-v2`) — semantic embeddings
- `scikit-learn` — Agglomerative Clustering (cosine distance)
- `TextBlob` — sentiment & subjectivity scoring
- `KeyBERT` — keyword/keyphrase extraction
- `gradio` — interactive demo UI
- `matplotlib` / `wordcloud` — visualization

## Architecture
```
RSS Feeds (multiple outlets)
        │
        ▼
  Fetch + Clean Text
        │
        ▼
 Sentence Embeddings (MiniLM)
        │
        ▼
 Agglomerative Clustering  ──►  groups articles about the SAME event
        │
        ▼
 Sentiment + Subjectivity (per article)
 Keyword Extraction (per article)
        │
        ▼
   Event Comparison Report  +  Gradio UI
```

> ⚠️ **Important honesty note:** the "bias label" attached to each outlet below (Left / Lean Left /
> Center / Lean Right / Right) is a simplified, static tag based on commonly cited public media-bias
> aggregators (e.g. AllSides-style categorizations). It is a rough educational label, not a
> scientific measurement, and it does not change based on the article content itself — only the
> *sentiment/subjectivity/keyword* analysis below is computed from the actual text. Keep that
> distinction in mind when you present this project.


In [ ]:
# Install dependencies (Colab-friendly, quiet install)
!pip install -q feedparser sentence-transformers scikit-learn keybert textblob wordcloud gradio matplotlib pandas numpy
!python -m textblob.download_corpora > /dev/null 2>&1
print("Dependencies installed.")


In [ ]:
import feedparser
import pandas as pd
import numpy as np
import re
import html
from datetime import datetime

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from keybert import KeyBERT

import matplotlib.pyplot as plt
from wordcloud import WordCloud

pd.set_option("display.max_colwidth", 120)


## Step 1 — Data Ingestion

We pull live headlines + summaries from several outlets using their public RSS feeds.
RSS is used instead of a scraper or paid API because it's free, fast, and doesn't
require login/keys — perfect for a Colab demo.

Each outlet is tagged with a **simplified bias label** so we can compare framing
across the political spectrum later. Feel free to add/remove/edit feeds — RSS URLs
occasionally change, so if one fails, just swap in an alternative.


In [ ]:
# Outlet -> (RSS URL, simplified bias label)
# Labels are rough, static, educational categorizations only.
RSS_FEEDS = {
    "BBC News":        ("http://feeds.bbci.co.uk/news/rss.xml",              "Center"),
    "Reuters World":   ("https://www.reutersagency.com/feed/?best-topics=top-news",  "Center"),
    "NPR News":        ("https://feeds.npr.org/1001/rss.xml",                "Lean Left"),
    "The Guardian":    ("https://www.theguardian.com/world/rss",             "Lean Left"),
    "CNN Top Stories": ("http://rss.cnn.com/rss/cnn_topstories.rss",         "Lean Left"),
    "Fox News":        ("https://moxie.foxnews.com/google-publisher/latest.xml", "Right"),
    "New York Post":   ("https://nypost.com/feed/",                          "Right"),
    "Al Jazeera":      ("https://www.aljazeera.com/xml/rss/all.xml",         "Center / International"),
}

def fetch_articles(feeds: dict) -> pd.DataFrame:
    """Pull entries from each RSS feed and return a tidy DataFrame."""
    rows = []
    for outlet, (url, bias_label) in feeds.items():
        try:
            parsed = feedparser.parse(url)
            for entry in parsed.entries:
                title = html.unescape(entry.get("title", "")).strip()
                summary = html.unescape(entry.get("summary", entry.get("description", ""))).strip()
                link = entry.get("link", "")
                published = entry.get("published", "")
                if title:
                    rows.append({
                        "outlet": outlet,
                        "bias_label": bias_label,
                        "title": title,
                        "summary": summary,
                        "link": link,
                        "published": published,
                    })
        except Exception as e:
            print(f"[WARN] Failed to fetch {outlet}: {e}")
    return pd.DataFrame(rows)

print("Fetching live articles from RSS feeds...")
df = fetch_articles(RSS_FEEDS)
print(f"Fetched {len(df)} articles from {df['outlet'].nunique()} outlets.")
df.head()


## Step 2 — Cleaning the Text

RSS summaries often contain leftover HTML tags, extra whitespace, or "Read more"
boilerplate. We strip that out and build one clean `text` field (title + summary)
that we'll embed in the next step.


In [ ]:
def clean_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", text)          # strip HTML tags
    text = re.sub(r"http\S+", " ", text)           # strip URLs
    text = re.sub(r"\s+", " ", text).strip()       # collapse whitespace
    return text

df["clean_summary"] = df["summary"].apply(clean_text)
df["text"] = (df["title"] + ". " + df["clean_summary"]).str.strip()

# Drop empty / near-empty rows and exact duplicates
df = df[df["text"].str.len() > 20].drop_duplicates(subset="text").reset_index(drop=True)
print(f"{len(df)} articles remain after cleaning.")
df[["outlet", "bias_label", "title"]].head(10)


## Step 3 — Semantic Embeddings

Just like in the paper search engine, we convert each article's text into a
384-dimensional vector using `all-MiniLM-L6-v2`. Articles about the *same event*
will end up close together in this vector space, even if the outlets used
completely different wording.


In [ ]:
print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Encoding articles...")
embeddings = model.encode(df["text"].tolist(), show_progress_bar=True, normalize_embeddings=True)
print("Embeddings shape:", embeddings.shape)


## Step 4 — Clustering Same-Event Articles

We use **Agglomerative Clustering with cosine distance** instead of KMeans here,
because KMeans needs you to pick the number of clusters up front — but we have no
idea how many distinct "events" are in today's news. Agglomerative clustering with
a **distance threshold** instead lets the algorithm decide how many groups to form:
articles closer than the threshold get merged into the same event cluster.

Since embeddings are already L2-normalized, cosine distance = `1 - cosine similarity`.


In [ ]:
DISTANCE_THRESHOLD = 0.35  # lower = stricter (fewer, tighter clusters). Tune this.

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=DISTANCE_THRESHOLD,
    metric="cosine",
    linkage="average",
)
df["cluster"] = clustering.fit_predict(embeddings)

# Keep only clusters where 2+ DIFFERENT outlets covered the same event —
# that's the whole point of this project (comparing framing across sources).
multi_outlet_clusters = (
    df.groupby("cluster")["outlet"].nunique()
    .loc[lambda s: s >= 2]
    .sort_values(ascending=False)
)

print(f"Total clusters found: {df['cluster'].nunique()}")
print(f"Clusters covered by 2+ outlets (same event, multiple sources): {len(multi_outlet_clusters)}")
print()
print("Top cross-outlet events today:")
multi_outlet_clusters.head(10)


## Step 5 — Sentiment, Subjectivity & Framing Keywords

For every article we compute:
- **Polarity** (-1.0 negative → +1.0 positive) — the emotional tone of the language
- **Subjectivity** (0.0 factual → 1.0 opinionated) — how opinion-laden the wording is
- **Top keyphrases** (via KeyBERT, reusing the same MiniLM model) — what the outlet chose to emphasize

Comparing these *within the same event cluster* is where the interesting signal shows up:
two outlets can report the same event with very different tone or emphasis.


In [ ]:
kw_model = KeyBERT(model=model)  # reuse the same embedding model for consistency & speed

def score_sentiment(text: str):
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

def extract_keywords(text: str, top_n: int = 5):
    try:
        kws = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words="english", top_n=top_n)
        return [k for k, _ in kws]
    except Exception:
        return []

print("Scoring sentiment & subjectivity...")
sentiments = df["text"].apply(score_sentiment)
df["polarity"] = sentiments.apply(lambda x: x[0])
df["subjectivity"] = sentiments.apply(lambda x: x[1])

print("Extracting framing keywords (this can take a minute for larger datasets)...")
df["keywords"] = df["text"].apply(lambda t: extract_keywords(t, top_n=5))

df[["outlet", "bias_label", "title", "polarity", "subjectivity", "keywords"]].head(10)


## Step 6 — Event Comparison Report

This is the core deliverable: pick a cross-outlet event cluster and see, side by
side, how each outlet framed it — headline, tone, subjectivity, and top keywords.


In [ ]:
def show_event_report(cluster_id: int):
    subset = df[df["cluster"] == cluster_id].sort_values("outlet")
    if subset.empty:
        print("No articles found for that cluster id.")
        return

    print("=" * 90)
    print(f"EVENT CLUSTER #{cluster_id}  |  Covered by {subset['outlet'].nunique()} outlet(s)")
    print("=" * 90)

    for _, row in subset.iterrows():
        print(f"\n[{row['outlet']}]  ({row['bias_label']})")
        print(f"Headline    : {row['title']}")
        print(f"Polarity    : {row['polarity']:+.2f}   (-1 negative ... +1 positive)")
        print(f"Subjectivity: {row['subjectivity']:.2f}   (0 factual ... 1 opinionated)")
        print(f"Keywords    : {', '.join(row['keywords'])}")
        print(f"Link        : {row['link']}")
        print("-" * 90)

    # Quick spread summary
    pol_spread = subset["polarity"].max() - subset["polarity"].min()
    subj_spread = subset["subjectivity"].max() - subset["subjectivity"].min()
    print(f"\nTone spread across outlets -> polarity range: {pol_spread:.2f} | subjectivity range: {subj_spread:.2f}")


# Auto-pick today's biggest cross-outlet story as an example
if len(multi_outlet_clusters) > 0:
    example_cluster_id = multi_outlet_clusters.index[0]
    show_event_report(example_cluster_id)
else:
    print("No cross-outlet clusters found right now — try lowering DISTANCE_THRESHOLD and re-running Step 4.")


## Step 7 — Visualizing Framing Differences

Two quick visuals for any event cluster:
1. A bar chart comparing sentiment polarity per outlet
2. Word clouds of framing keywords, grouped by bias label


In [ ]:
def plot_cluster_sentiment(cluster_id: int):
    subset = df[df["cluster"] == cluster_id]
    if subset.empty:
        print("No articles for that cluster.")
        return
    plt.figure(figsize=(8, 4))
    colors = ["#d62728" if p < 0 else "#2ca02c" for p in subset["polarity"]]
    plt.barh(subset["outlet"], subset["polarity"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Sentiment polarity (-1 negative ... +1 positive)")
    plt.title(f"Sentiment by outlet — Event Cluster #{cluster_id}")
    plt.tight_layout()
    plt.show()

def plot_cluster_wordclouds(cluster_id: int):
    subset = df[df["cluster"] == cluster_id]
    if subset.empty:
        print("No articles for that cluster.")
        return
    text_blob = " ".join([" ".join(kws) for kws in subset["keywords"]])
    if not text_blob.strip():
        print("Not enough keyword data to build a word cloud.")
        return
    wc = WordCloud(width=800, height=400, background_color="white").generate(text_blob)
    plt.figure(figsize=(8, 4))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Framing keywords — Event Cluster #{cluster_id}")
    plt.show()

if len(multi_outlet_clusters) > 0:
    plot_cluster_sentiment(example_cluster_id)
    plot_cluster_wordclouds(example_cluster_id)


## Step 8 — Interactive Gradio Demo

A simple dropdown UI: pick a cross-outlet event, see the full comparison report.
This is what makes the project demo-able with a shareable link instead of a CLI.


In [ ]:
import gradio as gr

def cluster_choice_label(cid):
    sample_title = df[df["cluster"] == cid]["title"].iloc[0]
    n_outlets = df[df["cluster"] == cid]["outlet"].nunique()
    return f"#{cid} ({n_outlets} outlets) — {sample_title[:70]}"

cluster_choices = {cluster_choice_label(cid): cid for cid in multi_outlet_clusters.index}

def gradio_report(choice_label):
    cid = cluster_choices[choice_label]
    subset = df[df["cluster"] == cid].sort_values("outlet")
    lines = [f"### Event Cluster #{cid} — covered by {subset['outlet'].nunique()} outlets\n"]
    for _, row in subset.iterrows():
        lines.append(
            f"**{row['outlet']}** ({row['bias_label']})\n\n"
            f"- Headline: {row['title']}\n"
            f"- Polarity: {row['polarity']:+.2f} | Subjectivity: {row['subjectivity']:.2f}\n"
            f"- Keywords: {', '.join(row['keywords'])}\n"
            f"- [Read article]({row['link']})\n"
        )
    return "\n---\n".join(lines)

if cluster_choices:
    demo = gr.Interface(
        fn=gradio_report,
        inputs=gr.Dropdown(choices=list(cluster_choices.keys()), label="Pick a cross-outlet news event"),
        outputs=gr.Markdown(),
        title="News Bias & Fact Clustering Engine",
        description="Compare how different outlets framed the same real-world event.",
    )
    demo.launch(share=True, debug=False)
else:
    print("No cross-outlet clusters available right now to launch the demo with.")


## Notes, Limitations & Next Steps

**Be upfront about these when you present this project — it makes it stronger, not weaker:**

- **Bias labels are static and simplified.** They describe the outlet in general, not the
  specific article. Two articles from the same outlet can differ a lot in tone.
- **Sentiment ≠ bias.** Polarity/subjectivity scores measure *language tone*, not political
  slant or factual accuracy. A negative-polarity article isn't necessarily "biased" — the
  underlying event might just be bad news.
- **Clustering quality depends on `DISTANCE_THRESHOLD`.** Too low → no cross-outlet clusters
  form. Too high → unrelated stories get merged. Worth showing you tuned this deliberately.
- **RSS feeds change.** If a feed 404s, swap in an alternative — it doesn't break the pipeline.

**Natural extensions:**
1. Swap TextBlob for a transformer-based sentiment model (e.g. `cardiffnlp/twitter-roberta-base-sentiment-latest`) for more nuance.
2. Track the same event cluster over multiple days to see how framing shifts as a story develops.
3. Add a stance/entailment model to detect actual factual disagreement between outlets, not just tone.
4. Replace RSS with NewsAPI or GDELT for broader, historical coverage instead of just "today."
